## Import

In [ ]:
import sys
sys.path.append('..')
import os

import numpy as np

import my_datasets
from MyEnsemble import MyIF, MyEIF, MyNNIF, MyDIF
from MyInterpreter import MyInterpreter

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

from scipy.stats import pearsonr  

import pandas as pd

## Dataset

In [ ]:
data_dict = my_datasets.load(
    dataset_name="bisect3d", 
    scale="u",
)

X = data_dict['X']
y = data_dict['y']
name = data_dict['name']

## Plot

In [ ]:
x1_feature_index = 0
x2_feature_index = 1
intervals = 10
norm_type = "min"
magnitude = False

model = MyNNIF(
    contamination = np.mean(y),
    random_state = 0,
)
interpretation = "kNN"

########################################################################################################################

title = "LFI Score Map"

X = np.hstack((X[:, x1_feature_index].reshape(-1, 1), X[:, x2_feature_index].reshape(-1, 1)))

feature_min = np.min(X)
feature_max = np.max(X)

step = (feature_max - feature_min) / intervals

# Build a regular 2D grid
x_vals = np.arange(feature_min, feature_max + step, step)
y_vals = np.arange(feature_min, feature_max + step, step)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
X_eval = np.column_stack([X_grid.ravel(), Y_grid.ravel()])

# Fit forest and compute LFIs
model.fit(X)
interpreter = MyInterpreter(method=interpretation, ensemble=model)
LFIs = np.array([interpreter.LFI(x) for x in X_eval])
scores = np.array(model.decision_function(X_eval))
LFIs = LFIs.reshape(X_grid.shape + (2,))  # shape = (Ny, Nx, 2)

# Compute ratio and magnitude
LFIs_ratio = np.log(LFIs[..., 0]) - np.log(LFIs[..., 1])
LFIs_magnitude = np.log(LFIs[..., 0])
LFIs_magnitude /= LFIs_magnitude.max()  # normalize to [0,1]

# Build truncated HSV colormap
base_colors = plt.cm.hsv(np.linspace(2/3, 1.0, 1024))
cmap = LinearSegmentedColormap.from_list("hsv_tail", base_colors)

if norm_type == 'max':
    center = 0.0
    max_dev = max(abs(LFIs_ratio.max() - center), abs(LFIs_ratio.min() - center))
    norm = TwoSlopeNorm(vmin=center - max_dev, vcenter=center, vmax=center + max_dev)
    title += " - max deviation"

if norm_type == 'min':
    center = 0.0
    min_dev = min(abs(LFIs_ratio.max() - center), abs(LFIs_ratio.min() - center))
    norm = TwoSlopeNorm(vmin=center - min_dev, vcenter=center, vmax=center + min_dev)
    title += " - min deviation"

# Map ratio to colors
ratio_norm = norm(LFIs_ratio)
colors = cmap(ratio_norm)
colors[..., -1] = LFIs_magnitude  # alpha = magnitude

# Plot
fig, ax = plt.subplots(figsize=(6, 5))
mesh = ax.pcolormesh(X_grid, Y_grid, LFIs_ratio, cmap=cmap, norm=norm)
if magnitude:
    mesh.set_array(None)
    mesh.set_facecolors(colors.reshape(-1, 4))
    mesh.changed()
    title += f" - alpha = LFI{x1_feature_index+1}"
ax.set_title(title)
ax.set_xlabel(f"Feature {x1_feature_index+1}")
ax.set_ylabel(f"Feature {x2_feature_index+1}")
plt.colorbar(mesh, ax=ax, label=f"LFI{x1_feature_index+1}/LFI{x2_feature_index+1}")
plt.tight_layout()
plt.show()


In [ ]:
scores = scores.reshape(-1)
LFIs_0 = LFIs[..., 0].reshape(-1)
LFIs_1 = LFIs[..., 1].reshape(-1)
LFIs = LFIs_0 + LFIs_1

In [ ]:
from scipy.stats import pearsonr   

C = pearsonr(scores, LFIs)

In [ ]:
print(C)

In [ ]:
model = MyDIF(
    contamination = np.mean(y),
    random_state = 0,
)
interpretation = "kNN"

########################################################################################################################

X_eval = np.random.uniform(low=np.min(X, axis=0), high=np.max(X, axis=0), size=(1000, X.shape[-1]))

# Fit forest and compute LFIs
model.fit(X)
interpreter = MyInterpreter(method=interpretation, ensemble=model)
LFIs = np.array([interpreter.LFI(x) for x in X_eval])
scores = np.array(model.decision_function(X_eval))

In [ ]:
scores = scores.reshape(-1)
LFIs = np.sum(LFIs, axis=2).reshape(-1)

In [ ]:
from scipy.stats import pearsonr  
pearsonr(scores, LFIs)

In [ ]:
def LFI_correlation(
    model_class,
    interpretation,
    random_state
):

    intervals = 10

    folder = model_class.__name__
    os.makedirs(folder, exist_ok=True) 
    
    for dataset_name in my_datasets.datasets_names:

        print(dataset_name)

        GFIs_mean = pd.read_csv(f'{folder}\{interpretation}\{dataset_name}\{dataset_name}_mean.csv', index_col=0).to_numpy().reshape(-1)
        ranking = np.argsort(-GFIs_mean)

        data_dict = my_datasets.load(dataset_name)
        X = data_dict['X']
        y = data_dict['y']

        X = np.hstack((X[:, ranking[0]].reshape(-1, 1), X[:, ranking[1]].reshape(-1, 1)))
        feature_min = np.min(X, axis=0)
        feature_max = np.max(X, axis=0)
        step_0 = (feature_max[0] - feature_min[0]) / intervals
        step_1 = (feature_max[1] - feature_min[1]) / intervals
        x_vals = np.arange(feature_min[0], feature_max[0] + step_0, step_0)
        y_vals = np.arange(feature_min[1], feature_max[1] + step_1, step_1)
        X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
        X_eval = np.column_stack([X_grid.ravel(), Y_grid.ravel()])

        model = model_class(contamination=np.mean(y), random_state=random_state)
        model.fit(X)

        interpreter = MyInterpreter(interpretation, model)
        LFIs = np.array([interpreter.LFI(x) for x in X_eval])
        scores = np.array(model.decision_function(X_eval))

        scores = scores.reshape(-1)
        LFIs = LFIs.reshape(-1, 2)
        LFIs = np.sum(LFIs, axis=1).reshape(-1)

        correlation, p = pearsonr(scores, LFIs)

        correlation = pd.DataFrame(np.array([[correlation]]))

        if not os.path.isdir(f'{folder}\{interpretation}\{dataset_name}'):
            os.makedirs(f'{folder}\{interpretation}\{dataset_name}')
        correlation.to_csv(f'{folder}\{interpretation}\{dataset_name}\{dataset_name}_correlation.csv')

In [ ]:
LFI_correlation(MyIF, "DIFFI", 0)

In [ ]:
LFI_correlation(MyIF, "kNN", 0)

In [ ]:
LFI_correlation(MyIF, "ExIFFI", 0)
LFI_correlation(MyIF, "d", 0)

In [ ]:
LFI_correlation(MyEIF, "ExIFFI", 0)

In [ ]:
LFI_correlation(MyEIF, "kNN", 0)

In [ ]:
LFI_correlation(MyEIF, "d", 0)

In [ ]:
LFI_correlation(MyNNIF, "kNN", 0)

In [ ]:
LFI_correlation(MyNNIF, "d", 0)

In [ ]:
LFI_correlation(MyDIF, "kNN", 0)

In [ ]:
LFI_correlation(MyDIF, "d", 0)